# 05 · Comprehensive Reproducibility and Verification

# Project: Integrating Moral Values in Turkish EFL Classrooms

These notebooks are a reproducible analysis companion for the supplied project materials. They do not invent participant-level records. Appendix E/F provide aggregate frequencies with N=20; separate questionnaire notes contain percentages without an explicit denominator and conflict with the appendices on some items. Treat each source stream separately. The main manuscript describes a larger mixed-methods sample (the supplied abstract is truncated), so these appendix counts must not be silently generalized to the manuscript sample.

Run from any working directory. Generated files go to `./analysis_outputs` relative to the current working directory. Python 3.9+; dependencies: pandas, numpy, matplotlib (Notebook 3 only).

## Scope
Run end-to-end checks on the appendix summary data, Wilson calculations, qualitative coding utility, and figure/data artifact expectations. This notebook does not validate underlying raw participant data because none were supplied as machine-readable records.

In [ ]:
from pathlib import Path
import hashlib, json, math
import pandas as pd
import numpy as np
OUT=Path('analysis_outputs'); OUT.mkdir(exist_ok=True)

## Input and aggregate integrity
Recreate the core aggregate tables in-memory so this verification notebook does not depend on execution order of other notebooks.

In [ ]:
teacher = pd.DataFrame([
 ('responsibility',14),('global_citizenship',14),('empathy',3),('respect',2),('honesty',1),
 ('curriculum_no',16),('curriculum_yes',4),('debates',9),('storytelling',6),('role_play',3),('group_work',2),
 ('student_resistance',10),('time_constraints',6),('lack_training',4)],columns=['category','count'])
expected_sums={'values':20,'curriculum':20,'techniques':20,'obstacles':20}
groups={'values':['global_citizenship','empathy','respect','honesty'],
        'curriculum':['curriculum_no','curriculum_yes'],
        'techniques':['debates','storytelling','role_play','group_work'],
        'obstacles':['student_resistance','time_constraints','lack_training']}
for key,names in groups.items():
    got=int(teacher.set_index('category').loc[names,'count'].sum())
    assert got==expected_sums[key],(key,got)
print('All Appendix E grouped frequency totals verified at 20.')

## Statistical verification
Recompute key CI values and confirm boundaries/monotonicity. Use N=20 only for Appendix E counts; no denominator is imputed for separate unsourced percentage claims.

In [ ]:
from statistics import NormalDist
z=NormalDist().inv_cdf(.975)
def wilson(x,n):
    p=x/n; d=1+z*z/n; c=(p+z*z/(2*n))/d
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return c-h,c+h
p=14/20; se=math.sqrt(p*(1-p)/20); lo,hi=wilson(14,20)
assert abs(p-.70)<1e-12 and abs(se-0.1024695077)<1e-9
assert abs(lo-0.4810)<.001 and abs(hi-0.8550)<.001
assert 0<=wilson(0,20)[0]<wilson(0,20)[1]<=1
assert 0<=wilson(20,20)[0]<wilson(20,20)[1]<=1
print(f'14/20: p={p:.3f}, SE={se:.4f}, Wilson95=({lo:.4f}, {hi:.4f})')

## Reproducibility metadata and artifact audit
Hash available notebooks as they exist in the current directory; this is a local integrity manifest, not a citation or source-data checksum. Missing companion notebooks are reported rather than treated as a hard error.

In [ ]:
notebooks=[Path(f'{i:02d}_{name}.ipynb') for i,name in [
 (1,'Data_Generation_and_Cleaning'),(2,'Wilson_Confidence_Intervals_and_SE'),
 (3,'Publication_Figures_Generation'),(4,'Qualitative_Thematic_Coding_and_Audit'),
 (5,'Comprehensive_Reproducibility_and_Verification')]]
manifest=[]
for pth in notebooks:
    if pth.exists():
        raw=pth.read_bytes(); doc=json.loads(raw)
        assert doc['nbformat']==4 and isinstance(doc.get('cells'),list)
        manifest.append({'file':pth.name,'sha256':hashlib.sha256(raw).hexdigest(),'cells':len(doc['cells'])})
    else: print('Not present from this working directory:',pth)
print(json.dumps(manifest,indent=2))

## Verification checklist / known limitations
- Aggregate counts and percentage arithmetic are checked; figures and CSVs should be regenerated by their source notebooks.
- Appendix tables report N=20; manuscript narrative points to a larger sample. Reconcile before external reporting.
- Student table totals are inferred from each item’s category sum.
- Notes include incompatible teacher-responsibility claims and percentages with no stated N.
- No raw response file, sampling frame, participant-level records, coder labels, or ethics approval evidence was supplied. Consequently no participant-level cleaning, confirmatory inference, or qualitative reliability claim can be verified.
- Run notebooks in order in a clean Python environment; archive environment/package versions and source-data checksums when primary data become available.